In [1]:
# Membuat dataset
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [2]:
# Membuat SparkSession dan membaca dataset
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas12_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset CSV
df = spark.read.csv(
    "tugas12_ulasan.csv",
    header=True,
    inferSchema=True
)

# Melihat struktur dan isi dataset
df.printSchema()
df.show(10, truncate=False)

print("Jumlah data:", df.count())

26/11/19 14:15:15 WARN Utils: Your hostname, yan-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/19 14:15:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 14:15:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|nama_menu     |cabang    |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|Kopi Susu     |Yogyakarta|Sore       |5  |5                 |1             |
|Es Teh        |Yogyakarta|Pagi       |1  |21                |0             |
|Cappuccino    |Yogyakarta|Sore       |5  |15                |0             |
|Kentang Goreng|Semarang  |Malam      |2  |5                 |1             |
|Americano     |Yogyakarta|Malam      |2  |10                |0             |
|Americano     |Yogyakarta|Pagi       |3  |10                |1             |
|Americano     |Yogyakarta|Siang    

In [3]:
# Membagi data training dan testing
# Membagi data 80:20 :
train, test = df.randomSplit(
    [0.8, 0.2],
    seed=77
)

print("Jumlah data training:", train.count())
print("Jumlah data testing:", test.count())

Jumlah data training: 2411
Jumlah data testing: 589


In [ ]:
# A. MEMBANGUN ML PIPELINE

# buat Pipeline dengan tahapan:

# StringIndexer → OneHotEncoder → VectorAssembler → LogisticRegression


In [4]:
# Membuat Pipeline Logistic Regression
from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.ml.classification import LogisticRegression

# 1. StringIndexer
indexer = StringIndexer(
    inputCols=[
        "nama_menu",
        "cabang",
        "waktu_pesan"
    ],
    outputCols=[
        "nama_menu_index",
        "cabang_index",
        "waktu_pesan_index"
    ]
)

# 2. OneHotEncoder
encoder = OneHotEncoder(
    inputCols=[
        "nama_menu_index",
        "cabang_index",
        "waktu_pesan_index"
    ],
    outputCols=[
        "nama_menu_ohe",
        "cabang_ohe",
        "waktu_pesan_ohe"
    ]
)

# 3. VectorAssembler
assembler = VectorAssembler(
    inputCols=[
        "nama_menu_ohe",
        "cabang_ohe",
        "waktu_pesan_ohe",
        "qty",
        "waktu_tunggu_menit"
    ],
    outputCol="features"
)

# 4. Logistic Regression
lr = LogisticRegression(
    featuresCol="features",
    labelCol="ulasan_positif"
)

# 5. Membuat Pipeline
pipeline_lr = Pipeline(
    stages=[
        indexer,
        encoder,
        assembler,
        lr
    ]
)

# Melatih model menggunakan data training
model_lr = pipeline_lr.fit(train)

print("Pipeline Logistic Regression berhasil dibuat.")

Pipeline Logistic Regression berhasil dibuat.


In [5]:
# B. PELATIHAN DAN EVALUASI 

# Melakukan prediksi pada data testing
predictions_lr = model_lr.transform(test)

# Menampilkan hasil prediksi
predictions_lr.select(
    "ulasan_positif",
    "prediction",
    "probability"
).show(10, truncate=False)

+--------------+----------+----------------------------------------+
|ulasan_positif|prediction|probability                             |
+--------------+----------+----------------------------------------+
|1             |1.0       |[0.30580926470532666,0.6941907352946733]|
|1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|1             |1.0       |[0.08601505227726715,0.9139849477227329]|
|1             |1.0       |[0.1291608175231565,0.8708391824768436] |
|0             |0.0       |[0.5224849058902362,0.4775150941097638] |
|1             |1.0       |[0.08772906207315541,0.9122709379268445]|
|1             |1.0       |[0.1911272298028174,0.8088727701971826] |
|1             |1.0       |[0.26921410010812313,0.7307858998918768]|
|0             |0.0       |[0.915674451330994,0.08432554866900599] |
|0             |0.0       |[0.9313120353886019,0.06868796461139814]|
+--------------+----------+----------------------------------------+
only showing top 10 rows



In [6]:
# Menghitung Accuracy, F1-score, dan AUC-ROC
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

# Evaluasi Accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif",
    predictionCol="prediction",
    metricName="accuracy"
)

# Evaluasi F1-score
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif",
    predictionCol="prediction",
    metricName="f1"
)

# Evaluasi AUC-ROC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="ulasan_positif",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Menghitung hasil evaluasi
accuracy_lr = evaluator_accuracy.evaluate(predictions_lr)
f1_lr = evaluator_f1.evaluate(predictions_lr)
auc_lr = evaluator_auc.evaluate(predictions_lr)

# Menampilkan hasil
print("HASIL EVALUASI LOGISTIC REGRESSION")
print("-----------------------------------")
print("Accuracy :", round(accuracy_lr, 4))
print("F1-Score :", round(f1_lr, 4))
print("AUC-ROC  :", round(auc_lr, 4))

HASIL EVALUASI LOGISTIC REGRESSION
-----------------------------------
Accuracy : 0.7929
F1-Score : 0.7926
AUC-ROC  : 0.865


In [7]:
# Membuat confusion matrix menggunakan groupBy
confusion_matrix_lr = predictions_lr.groupBy(
    "ulasan_positif",
    "prediction"
).count().orderBy(
    "ulasan_positif",
    "prediction"
)

print("CONFUSION MATRIX LOGISTIC REGRESSION")

confusion_matrix_lr.show()

CONFUSION MATRIX LOGISTIC REGRESSION
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [8]:
# C. MODEL PEMBANDING
# Membuat Pipeline Random Forest

from pyspark.ml.classification import RandomForestClassifier

# Membuat model Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="ulasan_positif",
    numTrees=30,
    seed=77
)

# Membuat Pipeline Random Forest
pipeline_rf = Pipeline(
    stages=[
        indexer,
        encoder,
        assembler,
        rf
    ]
)

# Melatih model
model_rf = pipeline_rf.fit(train)

print("Pipeline Random Forest berhasil dilatih.")

Pipeline Random Forest berhasil dilatih.


In [9]:
# Melakukan prediksi pada data testing
predictions_rf = model_rf.transform(test)

# Menampilkan hasil prediksi
predictions_rf.select(
    "ulasan_positif",
    "prediction",
    "probability"
).show(10, truncate=False)

+--------------+----------+----------------------------------------+
|ulasan_positif|prediction|probability                             |
+--------------+----------+----------------------------------------+
|1             |1.0       |[0.28322243353858806,0.7167775664614119]|
|1             |0.0       |[0.7268541481309437,0.2731458518690562] |
|1             |1.0       |[0.24690942670528543,0.7530905732947146]|
|1             |1.0       |[0.24869223017923134,0.7513077698207687]|
|0             |0.0       |[0.6026403962064834,0.3973596037935166] |
|1             |1.0       |[0.2528283158061426,0.7471716841938574] |
|1             |1.0       |[0.2498659708273295,0.7501340291726705] |
|1             |1.0       |[0.26881988986098476,0.7311801101390153]|
|0             |0.0       |[0.81023975786136,0.18976024213863996]  |
|0             |0.0       |[0.81023975786136,0.18976024213863996]  |
+--------------+----------+----------------------------------------+
only showing top 10 rows



In [10]:
# Evaluasi Random Forest
# Menghitung Accuracy
accuracy_rf = evaluator_accuracy.evaluate(predictions_rf)

# Menghitung F1-score
f1_rf = evaluator_f1.evaluate(predictions_rf)

# Menghitung AUC-ROC
auc_rf = evaluator_auc.evaluate(predictions_rf)

# Menampilkan hasil evaluasi
print("HASIL EVALUASI RANDOM FOREST")
print("----------------------------")
print("Accuracy :", round(accuracy_rf, 4))
print("F1-Score :", round(f1_rf, 4))
print("AUC-ROC  :", round(auc_rf, 4))

HASIL EVALUASI RANDOM FOREST
----------------------------
Accuracy : 0.7963
F1-Score : 0.7959
AUC-ROC  : 0.8455


In [11]:
# Membandingkan kedua model
# Membuat tabel perbandingan
hasil_perbandingan = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_lr,
        accuracy_rf
    ],
    "F1-Score": [
        f1_lr,
        f1_rf
    ],
    "AUC-ROC": [
        auc_lr,
        auc_rf
    ]
})

print("PERBANDINGAN HASIL MODEL")
print(hasil_perbandingan.to_string(index=False))

PERBANDINGAN HASIL MODEL
              Model  Accuracy  F1-Score  AUC-ROC
Logistic Regression  0.792869  0.792643 0.865009
      Random Forest  0.796265  0.795903 0.845495


In [12]:
# Menentukan model terbaik berdasarkan AUC-ROC
if auc_rf > auc_lr:
    nama_model_terbaik = "Random Forest"
    model_terbaik = model_rf
    prediksi_terbaik = predictions_rf

elif auc_lr > auc_rf:
    nama_model_terbaik = "Logistic Regression"
    model_terbaik = model_lr
    prediksi_terbaik = predictions_lr

else:
    # Jika AUC sama, gunakan F1-score sebagai pembanding
    if f1_rf > f1_lr:
        nama_model_terbaik = "Random Forest"
        model_terbaik = model_rf
        prediksi_terbaik = predictions_rf
    else:
        nama_model_terbaik = "Logistic Regression"
        model_terbaik = model_lr
        prediksi_terbaik = predictions_lr

print("MODEL TERBAIK:", nama_model_terbaik)

MODEL TERBAIK: Logistic Regression


In [13]:
# D. ANALISIS KESALAHAN MODEL 
# Mengambil data yang salah diprediksi
data_salah = prediksi_terbaik.filter(
    prediksi_terbaik.ulasan_positif !=
    prediksi_terbaik.prediction
)

# Menampilkan 10 baris kesalahan
print("10 DATA YANG SALAH DIPREDIKSI")

data_salah.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "waktu_tunggu_menit",
    "ulasan_positif",
    "prediction"
).show(10, truncate=False)

10 DATA YANG SALAH DIPREDIKSI
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |
|Americano|Magelang|Sore       |3  |13                |1             |0.0       |
|Americano|Magelang|Sore       |5  |6                 |0             |1.0       |
|Americano|Magelang|Sore       |5  |15                |1             |0.0       |
|Americano|Semarang|Malam      |1  |2                 |0            

In [14]:
# Menganalisis kesalahan berdasarkan waktu tunggu
# Mengelompokkan kesalahan berdasarkan waktu tunggu
analisis_kesalahan = data_salah.groupBy(
    "waktu_tunggu_menit"
).count().orderBy(
    "waktu_tunggu_menit"
)

print("ANALISIS KESALAHAN BERDASARKAN WAKTU TUNGGU")

analisis_kesalahan.show()

ANALISIS KESALAHAN BERDASARKAN WAKTU TUNGGU
+------------------+-----+
|waktu_tunggu_menit|count|
+------------------+-----+
|                 2|    1|
|                 3|    3|
|                 4|    2|
|                 5|    2|
|                 6|    6|
|                 7|    5|
|                 8|    7|
|                 9|    6|
|                10|    8|
|                11|   11|
|                12|   10|
|                13|   10|
|                14|   13|
|                15|    6|
|                16|    8|
|                17|    6|
|                18|    2|
|                19|    4|
|                20|    2|
|                21|    1|
+------------------+-----+
only showing top 20 rows



In [15]:
# Menghitung jumlah kesalahan
# Menghitung jumlah data yang salah
jumlah_salah = data_salah.count()

# Menghitung jumlah data testing
jumlah_testing = test.count()

# Menghitung persentase kesalahan
persentase_salah = (
    jumlah_salah / jumlah_testing * 100
    if jumlah_testing > 0 else 0
)

print("Jumlah data testing:", jumlah_testing)
print("Jumlah prediksi salah:", jumlah_salah)
print("Persentase kesalahan:", round(persentase_salah, 2), "%")

Jumlah data testing: 589
Jumlah prediksi salah: 122
Persentase kesalahan: 20.71 %


In [16]:
# penutup
# Menghentikan SparkSession

spark.stop()
print("Seluruh pengerjaan Tugas Mandiri selesai.")

Seluruh pengerjaan Tugas Mandiri selesai.
